# **Solemne 1 — Minería de Datos 2026**
## ¿qué espero encontrar?
En la tabla de 'health_nutrition_population' espero encontrar referencias a la salud nutricional de la poblacion en algun pais/ciudad especificado en la tabla.

In [12]:
from google.cloud import bigquery
import pandas as pd

client = bigquery.Client(project="yo-si-489418")

dataset_ref = client.dataset("world_bank_health_population", project="bigquery-public-data") #Dentro del la bigquery publica, busca el proyecto del banco mundial

tables = list(client.list_tables(dataset_ref))

table_ref = dataset_ref.table("health_nutrition_population")

query = """
SELECT
  country_name,
  indicator_name,
  value,
  year
FROM `bigquery-public-data.world_bank_health_population.health_nutrition_population`
WHERE `country_name` IN (
  'Argentina', 'Bolivia', 'Brasil', 'Chile', 'Colombia',
  'Costa Rica', 'Cuba', 'Ecuador', 'El Salvador', 'Guatemala',
  'Honduras', 'México', 'Nicaragua', 'Panamá', 'Paraguay',
  'Perú', 'República Dominicana', 'Uruguay', 'Venezuela'
)
  AND value IS NOT NULL
  AND year >= 2000
ORDER BY year DESC
"""


dry_run_config = bigquery.QueryJobConfig(dry_run=True)

dry_run_query_job = client.query(query, job_config=dry_run_config) #Estas lineas de codigo me mostraran cuando pesaran los datos que estare consultando.

print()
print("This query will process {} bytes.".format(dry_run_query_job.total_bytes_processed))



safe_config = bigquery.QueryJobConfig(maximum_bytes_billed=10**10) #Esto limita la cantidad de bytes que podria procesar (quizas no necesario pero no esta de mas)
query_job = client.query(query, job_config=safe_config)


Pandass = query_job.to_dataframe()       #transforma la query a un dataframe de pandas

Pandass

c:\Users\joshe\Documents\U weas\GIT\venv\Lib\site-packages\google\auth\_default.py:114: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)



This query will process 239235925 bytes.


,country_name,indicator_name,value,year
0,Paraguay,Children orphaned by HIV/AIDS,1.000000,2020
1,Argentina,"Population ages 75-79, male",1.000000,2020
2,El Salvador,"Age population, age 22, male, interpolated",1.000000,2020
3,Honduras,"Population ages 35-39, female (% of female pop...",6.959987,2020
4,El Salvador,"Population ages 75-79, female (% of female pop...",1.829017,2020
...,...,...,...,...
84874,Paraguay,"Incidence of tuberculosis (per 100,000 people)",1.000000,2000
84875,Paraguay,Antiretroviral therapy coverage for PMTCT (% o...,1.000000,2000
84876,Cuba,"Cause of death, by non-communicable diseases (...",81.252044,2000
84877,Chile,"Population ages 45-49, female",1.000000,2000


# Analisis de Datos

In [13]:
print(f"Filas: {Pandass.shape[0]:,} | Columnas: {Pandass.shape[1]}")
print("Tipos:", ", ".join([f"{col}: {dtype}" for col, dtype in Pandass.dtypes.items()]))
print(f"Rango temporal: {Pandass['year'].min()} - {Pandass['year'].max()}")

nulos = Pandass.isnull().sum()
print("Nulos:", ", ".join([f"{col}: {val:,}" for col, val in nulos.items()]))

Filas: 84,879 | Columnas: 4
Tipos: country_name: object, indicator_name: object, value: float64, year: Int64
Rango temporal: 2000 - 2020
Nulos: country_name: 0, indicator_name: 0, value: 0, year: 0


 Viendo estos datos los cuales fueron filtrado a datos mas actuales (Del 2000 en adelante) esto para tener una mayor fiabilidad en la compilacion de estos (tambien porque no existen nulos), espero que tengan una calidad decente para poder analisarlos de forma optima. Con un alcance directamente sobre la poblacion chilena, como la cantidad de filas no es tan grande podria ser que los datos a analizar no nos den una estadistica exacta.

In [ ]:
df_suicidios = Pandass[Pandass['indicator_name'].str.contains('suicide', case=False, na=False)]

df_so = df_suicidios.sort_values("country_name")

print(f"Filas: {df_so.shape[0]:,}")

df_so

Filas: 780


,country_name,indicator_name,value,year
11367,Argentina,"Suicide mortality rate, female (per 100,000 fe...",3.5,2017
42766,Argentina,"Suicide mortality rate (per 100,000 population)",8.6,2010
5967,Argentina,"Suicide mortality rate, female (per 100,000 fe...",3.3,2019
10063,Argentina,"Suicide mortality rate, male (per 100,000 male...",15.1,2018
42641,Argentina,"Suicide mortality rate, female (per 100,000 fe...",3.2,2010
...,...,...,...,...
61583,Uruguay,"Suicide mortality rate, female (per 100,000 fe...",6.7,2005
61584,Uruguay,"Suicide mortality rate (per 100,000 population)",14.8,2005
61620,Uruguay,"Suicide mortality rate, male (per 100,000 male...",23.4,2005
60386,Uruguay,"Suicide mortality rate, male (per 100,000 male...",23.7,2006
